# Training MetricGAN+KAN
Mainly based on train.py from MetricGAN+.

In [ ]:
!pip install speechbrain
!pip install hyperpyyaml
!pip install pesq
!pip install pysepm
!pip install git+https://github.com/schmiph2/pysepm.git
!pip install tensorboard

In [ ]:
import os
import shutil
import copy
import sys
from enum import Enum, auto

import torch
import torchaudio
from hyperpyyaml import load_hyperpyyaml
from pesq import pesq

import speechbrain as sb
from speechbrain.dataio.sampler import ReproducibleWeightedRandomSampler
from speechbrain.nnet.loss.stoi_loss import stoi_loss
from speechbrain.processing.features import spectral_magnitude
from speechbrain.utils.distributed import run_on_main
from speechbrain.utils.metric_stats import MetricStats

# from MetricGAN_KAN import MetricDiscriminator, EnhancementGenerator

from pysepm import composite
from train import *
from spectrogram import *

N_fft = 512
data_folder = "../data/noisy-vctk-16k"
output_folder = "./results"
figures_folder = "./figures"

In [ ]:
class PerceptualDistillation:
    def __init__(self, generator, sample_rate=16000):
        self.generator_old = copy.deepcopy(generator)
        self.generator_old.eval()
        for p in self.generator_old.parameters():
            p.requires_grad_(False)
        self.sample_rate = sample_rate

    def distillation_loss(self, noisy_wavs, lens, new_generator, hparams):
        # Feature pipeline identical to MGKBrain.compute_feats
        feats = hparams.compute_STFT(noisy_wavs)
        feats = spectral_magnitude(feats, power=0.5)
        feats = torch.log1p(feats)

        with torch.no_grad():
            mask_old = self.generator_old(feats, lengths=lens)
            mask_old = mask_old.clamp(min=hparams.min_mask).squeeze(2)
            spec_old = torch.mul(mask_old, feats)
            wav_old = hparams.resynth(torch.expm1(spec_old), noisy_wavs).detach()

        mask_new = new_generator(feats, lengths=lens)
        mask_new = mask_new.clamp(min=hparams.min_mask).squeeze(2)
        spec_new = torch.mul(mask_new, feats)
        wav_new = hparams.resynth(torch.expm1(spec_new), noisy_wavs)

        return stoi_loss(wav_new, wav_old, lens)

In [ ]:
class MGKBrainPD(MGKBrain):
    def __init__(self, *args, pd_lambda=0.5, pd_anchor_interval=50, **kwargs):
        super().__init__(*args, **kwargs)
        self.perceptual_distillation = None
        self.pd_lambda = pd_lambda
        self.pd_anchor_interval = pd_anchor_interval

    def on_stage_start(self, stage, epoch=None):
        super().on_stage_start(stage, epoch)
        if stage == sb.Stage.TRAIN and self.pd_lambda != 0:
            if self.perceptual_distillation is None or epoch % self.pd_anchor_interval == 1:
                self.perceptual_distillation = PerceptualDistillation(
                    self.modules.generator
                )

    def compute_objectives(self, predictions, batch, stage, optim_name=""):
        loss = super().compute_objectives(predictions, batch, stage, optim_name=optim_name)
        if (
            self.pd_lambda != 0
            and stage == sb.Stage.TRAIN
            and self.sub_stage == SubStage.GENERATOR
            and self.perceptual_distillation is not None
        ):
            noisy_wavs, lens = batch.noisy_sig
            pd_loss = self.perceptual_distillation.distillation_loss(
                noisy_wavs, lens, self.modules.generator, self.hparams
            )
            self.hparams.train_logger.log_stats({"PD loss (unscaled)": pd_loss.item()})
            loss = loss + self.pd_lambda * pd_loss
        return loss

# Requirements
```bash
pip install speechbrain
pip install https://github.com/schmiph2/pysepm/archive/master.zip
```

If `kaiser` is not found leading to an `ImportError`, you may need to go to line 2 of /path/to/your/python_env/site-packages/pysepm/utils.py, and change
```python
from scipy.signal import firls,kaiser,upfirdn
```
to
```python
from scipy.signal import firls,upfirdn
from scipy.signal.windows import kaiser
```

# Training
Mainly based on train.py from MetricGAN+.

In [ ]:
    se_brain.train_set = datasets["train"]
    se_brain.batch_size = hparams["dataloader_options"]["batch_size"]
    se_brain.sub_stage = SubStage.GENERATOR
    se_brain.historical_set = {}
    se_brain.noisy_scores = {}

    if not os.path.isfile(hparams["historical_file"]):
        shutil.rmtree(hparams["MetricGAN_KAN_folder"], ignore_errors=True)
    run_on_main(create_folder, kwargs={"folder": hparams["MetricGAN_KAN_folder"]})

    se_brain.load_history()

Loop through the model files and train them.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

task_yamls = ["hparams/train_mgk_g4_d4.yaml", "hparams/train_mgk_g4_d4_lwf_off.yaml"]

# Set one GPU id per task for parallel execution, e.g. [0, 1].
# If this is empty or too short, training falls back to sequential mode.
GPU_IDS = [0, 1]

def _run_one(yaml_file, gpu_id):
    run_opts = {"device": f"cuda:{gpu_id}"} if gpu_id is not None else None
    print(f"Training: {yaml_file} on {run_opts['device'] if run_opts else 'default device'}")
    return yaml_file, train_and_eval(yaml_file, run_opts_override=run_opts)

results = {}
can_parallel = len(GPU_IDS) >= len(task_yamls) and torch.cuda.device_count() >= len(task_yamls)

if can_parallel:
    print(f"Parallel mode with {len(task_yamls)} workers")
    with ThreadPoolExecutor(max_workers=len(task_yamls)) as ex:
        futures = [
            ex.submit(_run_one, yaml_file, GPU_IDS[idx])
            for idx, yaml_file in enumerate(task_yamls)
        ]
        for fut in as_completed(futures):
            yaml_file, result = fut.result()
            results[yaml_file] = result
else:
    print("Sequential fallback (not enough configured GPUs for safe parallel training)")
    for yaml_file in task_yamls:
        _, result = _run_one(yaml_file, None)
        results[yaml_file] = result

brains = {k: v[0] for k, v in results.items()}
datasets_map = {k: v[1] for k, v in results.items()}

In [ ]:
import glob
import re
import matplotlib.pyplot as plt


def _extract_metric(line, keys):
    for key in keys:
        m = re.search(rf"{key}\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line, flags=re.IGNORECASE)
        if m:
            return float(m.group(1))
    return None


def load_training_curve(log_file):

    epochs, valid_pesq, valid_csig, valid_cbak, valid_covl = [], [], [], [], []

    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if "epoch" not in line.lower():
                continue

            m_epoch = re.search(r"epoch\s*[:=]\s*(\d+)", line, flags=re.IGNORECASE)
            if not m_epoch:
                continue

            epochs.append(int(m_epoch.group(1)))
            valid_pesq.append(_extract_metric(line, [r"pesq", r"valid[_\s-]*pesq"]))
            valid_csig.append(_extract_metric(line, [r"csig", r"valid[_\s-]*csig"]))
            valid_cbak.append(_extract_metric(line, [r"cbak", r"valid[_\s-]*cbak"]))
            valid_covl.append(_extract_metric(line, [r"covl", r"valid[_\s-]*covl"]))

    return {
        "epoch": epochs,
        "valid_pesq": valid_pesq,
        "valid_csig": valid_csig,
        "valid_cbak": valid_cbak,
        "valid_covl": valid_covl,
    }


candidates = sorted(glob.glob(f"{output_folder}/**/train_log.txt", recursive=True))
if not candidates:
    raise FileNotFoundError("Kein train_log.txt gefunden. Trainiere zuerst mindestens 1 Run.")

log_file = candidates[-1]
curves = load_training_curve(log_file)

if len(curves["epoch"]) == 0:
    raise RuntimeError(f"Keine Epochenzeilen in {log_file} gefunden.")

fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=120)
plots = [
    ("valid_pesq", "Valid PESQ"),
    ("valid_csig", "Valid CSIG"),
    ("valid_cbak", "Valid CBAK"),
    ("valid_covl", "Valid COVL"),
]

for ax, (key, title) in zip(axes.flat, plots):
    y = curves[key]
    x = [e for e, v in zip(curves["epoch"], y) if v is not None]
    y = [v for v in y if v is not None]

    if len(x) > 0:
        ax.plot(x, y, marker="o", linewidth=1.5, markersize=3)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Training Curves ({os.path.dirname(log_file)})")
plt.tight_layout()

os.makedirs(figures_folder, exist_ok=True)
out_png = os.path.join(figures_folder, "training_curves_latest.png")
plt.savefig(out_png, bbox_inches="tight")
plt.show()

print(f"Log-Datei: {log_file}")
print(f"Diagramm gespeichert: {out_png}")

Generate spectrograms.

In [ ]:
model_name_list = [fname.split(".")[-2] for fname in os.listdir("models") if fname.endswith(".yaml")]
enh_folder = [f"{output_folder}/{fname.split(".")[-2]}/enhanced_wavs" for fname in os.listdir("models") if fname.endswith(".yaml")]
# os.path.join(output_folder, model_name)
for m in model_name_list:
    target_dir = f"{figures_folder}/{m}"
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
    save_spec(f"{output_folder}/{m}/enhanced_wavs", target_dir)

Codes for generating model's hyper-param files.

In [ ]:
def generate_train_config():
    templates = None
    with open("hparams/train.yaml") as f:
        templates = f.readlines()
    for fname in os.listdir("models"):
        tmp = fname.split(".")
        if tmp[-1] == "yaml":
            with open(f"hparams/train_{tmp[-2]}.yaml", "w") as f:
                spec = templates.copy()
                spec[64] = f"models: !include:../models/{fname}\n"
                f.write("# Auto generated basing on train.yaml\n")
                f.writelines(spec)
            
# generate_train_config()